In [1]:

import torch
import numpy as np
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import Subset
import roboticstoolbox as rtb
from spatialmath import SE3
from typing import Tuple
import torch.nn.functional as F
import math


from models import TConvGNN, JointMLP, IK_CVAE, IK_CVAE_GraphEnc_MLPDec



In [2]:
# Проверяем наличие CUDA
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Используется GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA не найдена, используется CPU")

Используется GPU: NVIDIA GeForce RTX 3060


In [3]:
data, slices = torch.load('dataset/panda_dataset/processed/data.pt', weights_only=False)

class LoadedDataset(InMemoryDataset):
    def __init__(self, data, slices):
        super().__init__()
        self.data = data
        self.slices = slices
        self._num_samples = self.data.x.shape[0] if hasattr(self.data, 'x') else None

dataset = LoadedDataset(data, slices)

print(f"Всего примеров: {len(dataset)}")
print(dataset[0])  # первый граф

Всего примеров: 120000
Data(x=[8, 4], edge_index=[2, 26], edge_attr=[26, 5], y=[7], global_feat=[1, 6])


c:\ProgramData\miniconda3\envs\robotics\lib\site-packages\torch_geometric\data\in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [4]:
from torch.utils.data import Subset

# Разделим индексы (например, 80/10/10)
num_samples = len(dataset)
indices = torch.randperm(num_samples)
train_idx = indices[:int(0.8*num_samples)]
valid_idx   = indices[int(0.8*num_samples):int(0.9*num_samples)]
test_idx  = indices[int(0.9*num_samples):]

train_dataset = Subset(dataset, train_idx)
valid_dataset   = Subset(dataset, valid_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader   = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
q_start_all, q_lims_all, dh_all = [], [], []
pos_all, rpy_all = [], []
for data in train_dataset:
    real_x = data.x[:-1]                              # (7,4)
    q_start_all.append(real_x[:, 0])                 # углы
    q_lims_all.append(real_x[:, 2:4])                # q_min, q_max
    # рёбра цепи (тип 0)
    chain_edges = data.edge_attr[data.edge_attr[:, -1] == 0]
    dh_all.append(chain_edges[:, :4])                # a, alpha, d, offset
    gf = data.global_feat.squeeze(0)  # (6,)
    pos_all.append(gf[:3])
    rpy_all.append(gf[3:6])

q_start_all = torch.cat(q_start_all)                  # (N,)
q_lims_all  = torch.cat(q_lims_all)                   # (N,2)
dh_all      = torch.cat(dh_all)                       # (M,4)

q_mean = q_start_all.mean()
q_std  = q_start_all.std().clamp(min=1e-6)
lim_mean = q_lims_all.mean()
lim_std  = q_lims_all.std().clamp(min=1e-6)
dh_mean  = dh_all.mean(dim=0)                         # (4,)
dh_std   = dh_all.std(dim=0).clamp(min=1e-6)

    
pos_all = torch.stack(pos_all)   # (N, 3)
rpy_all = torch.stack(rpy_all)   # (N, 3)

pos_mean = pos_all.mean(dim=0)
pos_std  = pos_all.std(dim=0).clamp(min=1e-6)
rpy_mean = rpy_all.mean(dim=0)
rpy_std  = rpy_all.std(dim=0).clamp(min=1e-6)

In [ ]:
robot=rtb.models.DH.Panda()
# model = TConvGNN(hidden_dim=256, max_nodes=10, num_layers=3, dropout=0.1, heads=8)
# model = JointMLP(hidden_dim=256, num_layers=3, dropout=0.1)
# model = IK_CVAE(latent_dim=64, hidden_dim=256, num_layers=6, dropout=0.1, dec_heads=8, dec_lay=4)
model = IK_CVAE_GraphEnc_MLPDec(latent_dim=64, hidden_dim=512, heads=8)

opt = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-6)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100, eta_min=1e-5)

def cvae_loss(q_recon, q_goal, mu, logvar, beta=0.5):
    recon = F.mse_loss(q_recon, q_goal)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.shape[0]
    return recon + beta * kl, recon.item(), kl.item()

criterion = cvae_loss
scale = 1

def pose_metric(batch, q_pred_batch, robot):
    """
    batch: PyG Batch (нужен для извлечения global_feat)
    q_pred_batch: torch.Tensor (B, 7) – предсказанные q_goal
    robot: rtb.models.DH.Panda()
    """
    pos_sum, rot_sum = 0.0, 0.0
    B = batch.num_graphs

    for i in range(B):
        q_pred = q_pred_batch[i].cpu().numpy()
        global_feat = batch.global_feat[i].cpu().numpy()
        T_target = SE3(global_feat[:3]) * SE3.RPY(global_feat[3:6], order='zyx')

        T_pred = robot.fkine(q_pred)
        if T_pred is None:
            pos_sum += 10.0
            rot_sum += np.pi
            continue

        pos_err = np.abs(T_target.t - T_pred.t)
        pos_sum += np.mean(pos_err)

        rpy_pred = T_pred.rpy(order='zyx')
        rot_err = global_feat[3:6] - rpy_pred
        rot_err = np.abs(rot_err)
        rot_err = np.minimum(rot_err, 2*np.pi - rot_err)
        rot_sum += np.mean(rot_err)

    return pos_sum / B, rot_sum / B

In [ ]:
def train_step():
    model.train()
    running_loss = 0.0

    beta = 0.4

    for batch_data in train_loader:
        batch_data = batch_data.to(device)
        
        
        gf = batch_data.global_feat
        gf[:, :3] = gf[:, :3] - pos_mean.to(device)
        gf[:, 3:6] = gf[:, 3:6] - rpy_mean.to(device)
        
        ptr = batch_data.ptr
        B = batch_data.num_graphs
        q_goal_list = []
        for i in range(B):
            start = ptr[i]
            idx = torch.arange(start, start+7, device=device)
            q_start = batch_data.x[idx, 0]
            dq_true = batch_data.y[i*7 : i*7+7]
            q_goal_list.append(q_start + dq_true)
        q_goal_batch = torch.stack(q_goal_list)
        
        opt.zero_grad()
        q_recon, mu, logvar = model(batch_data, q_goal_batch)
        loss, _, _ = cvae_loss(q_recon, q_goal_batch, mu, logvar, beta=beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)


def val_step():
    model.eval()
    running_loss = 0.0
    running_pos, running_rot = 0.0, 0.0
    beta = 0.4
    with torch.no_grad():
        for batch_data in valid_loader:
            batch_data = batch_data.to(device)

            orig_global_feat = batch_data.global_feat.clone()

            gf = batch_data.global_feat
            gf[:, :3] = gf[:, :3] - pos_mean.to(device)
            gf[:, 3:6] = gf[:, 3:6] - rpy_mean.to(device)

            ptr = batch_data.ptr
            B = batch_data.num_graphs
            q_goal_true_list = []
            for i in range(B):
                start = ptr[i]
                idx = torch.arange(start, start+7, device=device)
                q_start = batch_data.x[idx, 0]
                dq_true = batch_data.y[i*7 : i*7+7]
                q_goal_true_list.append(q_start + dq_true)
            q_goal_true_batch = torch.stack(q_goal_true_list)

            q_recon, mu, logvar = model(batch_data, q_goal_true_batch)
            loss, _, _ = cvae_loss(q_recon, q_goal_true_batch, mu, logvar, beta=beta)
            running_loss += loss

            q_pred = model(batch_data, q_goal=None)
            batch_data.global_feat = orig_global_feat
            pos_mae, rot_mae = pose_metric(batch_data, q_pred, robot)
            running_pos += pos_mae
            running_rot += rot_mae

    n = len(valid_loader)
    return running_loss.item()/n, running_pos/n, running_rot/n

In [ ]:
train_losses = []
valid_losses = []
pose_mae = []
rot_mae = []

In [ ]:
from tqdm import tqdm

best_metric = 0.1155
epoch = 0

In [ ]:
epochs = 15
model.to(device)


for _ in (pbar := tqdm(range(epochs))):
    train_loss = train_step()
    valid_loss, pos_mae_new, rot_mae_new  = val_step()

    scheduler.step()
    
    pose_mae.append(pos_mae_new)
    rot_mae.append(rot_mae_new)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    epoch += 1
    print(train_loss, valid_loss, pos_mae_new, rot_mae_new)
    
    pbar.set_description(f'Avg. train/valid loss: {train_loss:.4f}/{valid_loss:.4f}/{pos_mae_new:.4f}/{rot_mae_new:.4f}')
    if pos_mae_new  < best_metric:
            torch.save(model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict(), 'best_model.pth')
            best_metric = pos_mae_new
            print(f'  Сохранена модель {epoch} эпохи')

Avg. train/valid loss: 1.0063/0.8571/0.1429/0.4961:   1%|          | 1/100 [00:21<35:55, 21.78s/it]

1.0062722495396932 0.8570779029359209 0.14289993024178047 0.496073738394998


Avg. train/valid loss: 0.8695/0.8496/0.1433/0.4811:   2%|▏         | 2/100 [00:44<36:16, 22.21s/it]

0.869453507900238 0.8496355097344581 0.14332965881356952 0.48105450730607496


Avg. train/valid loss: 0.8597/0.8409/0.1416/0.4986:   3%|▎         | 3/100 [01:05<34:58, 21.64s/it]

...

0.7557878470420838 0.7437789592337101 0.13612966186075506 0.49534661011017433


Avg. train/valid loss: 0.7558/0.7452/0.1355/0.4995:  44%|████▍     | 44/100 [16:33<21:20, 22.87s/it]

0.755803614616394 0.7452304109613946 0.13552332836061318 0.4994671519051596


Avg. train/valid loss: 0.7558/0.7452/0.1355/0.4995:  44%|████▍     | 44/100 [16:49<21:25, 22.95s/it]


KeyboardInterrupt: 

In [ ]:
# model.load_state_dict(torch.load('best_gat_model.pth', map_location=device))

In [ ]:
import time
import numpy as np
import torch
from spatialmath import SE3
import roboticstoolbox as rtb
from torch_geometric.loader import DataLoader

def test_pure_ik(loader, robot, device):
    """
    Чистый IK: для каждого примера последовательно запускаем ikine_LM от q_start.
    Замеряем полное время на один пример: извлечение данных, FK для цели, сам IK.
    """
    total_time = 0.0
    success = 0
    fail = 0
    total_examples = 0

    for batch_data in loader:  # batch_size=1 в тестовом загрузчике
        # Данные уже на CPU, так как загрузчик выдаёт один граф
        # Извлекаем необходимые данные
        global_feat_np = batch_data.global_feat.cpu().numpy()[0]  # (6,)
        ptr = batch_data.ptr.cpu().numpy()
        x = batch_data.x.cpu().numpy()
        t0 = time.perf_counter()
        start = ptr[0]
        q_start = x[start:start+7, 0].copy()  # (7,)

        # Целевая поза
        T_target = SE3(global_feat_np[:3]) * SE3.RPY(global_feat_np[3:6], order='zyx')

        
        sol = robot.ikine_LM(T_target, q0=q_start, ilimit=200, tol=1e-4, joint_limits=True)
        dt = time.perf_counter() - t0
        total_time += dt
        total_examples += 1
        if sol.success:
            success += 1
        else:
            fail += 1

    return total_time, success, fail, total_examples


def test_model_ik(loader, model, robot, pos_mean, rpy_mean, device):
    """
    Гибридный подход: модель + IK (с ограниченным числом итераций).
    Замеряем полное время на один пример:
      - подготовка данных (центрирование, перенос на GPU)
      - инференс модели (один пример)
      - перенос предсказания обратно на CPU
      - запуск IK с q0 = предсказание модели
    """
    model.eval()
    total_time = 0.0
    success = 0
    fail = 0
    total_examples = 0

    # Перемещаем статистики на нужное устройство
    pos_mean = pos_mean.to(device)
    rpy_mean = rpy_mean.to(device)

    with torch.no_grad():
        for batch_data in loader:  # batch_size=1
            # Исходные данные (на CPU)
            global_feat_orig = batch_data.global_feat.clone()  # (1, 6)
            ptr = batch_data.ptr
            x = batch_data.x

            # Переносим данные на GPU для модели
            t0 = time.perf_counter()
            batch_data = batch_data.to(device)

            # Подготовка для модели: центрирование global_feat
            gf = batch_data.global_feat
            gf[:, :3] -= pos_mean
            gf[:, 3:6] -= rpy_mean

            # Замеряем время с этого момента (включая центрирование и инференс)
            

            # Инференс модели (батч размером 1)
            q_pred = model(batch_data, q_goal=None)  # (1, 7)

            # Перенос предсказания на CPU (для использования в IK)
            q_pred_cpu = q_pred.cpu().numpy()[0]

            # Восстанавливаем оригинальную целевую позу (без центрирования)
            global_feat_np = global_feat_orig.cpu().numpy()[0]
            T_target = SE3(global_feat_np[:3]) * SE3.RPY(global_feat_np[3:6], order='zyx')

            # Запускаем IK с предсказанием как начальной точкой
            sol = robot.ikine_LM(T_target, q0=q_pred_cpu, ilimit=200, tol=1e-4, joint_limits=True)

            dt = time.perf_counter() - t0
            total_time += dt
            total_examples += 1
            if sol.success:
                success += 1
            else:
                fail += 1

    return total_time, success, fail, total_examples


# -------------------------------------------------------------------
# Пример использования
# -------------------------------------------------------------------




    # 1) Чистый IK
t_pure, suc_pure, fail_pure, n_pure = test_pure_ik(test_loader, robot, device)
avg_pure = t_pure / n_pure * 1000  # в миллисекундах

    # 2) Модель + IK
t_model, suc_model, fail_model, n_model = test_model_ik(test_loader, model, robot, pos_mean, rpy_mean, device)
avg_model = t_model / n_model * 1000

print(f"Чистый IK: всего {n_pure}, успешных {suc_pure}, провалов {fail_pure}, среднее время {avg_pure:.2f} мс")
print(f"Модель+IK: всего {n_model}, успешных {suc_model}, провалов {fail_model}, среднее время {avg_model:.2f} мс")
print(f"Ускорение: {(1 - avg_model/avg_pure)*100:.1f}%")

Чистый IK: всего 12000, успешных 11994, провалов 6, среднее время 0.02 мс
Модель+IK: всего 12000, успешных 11992, провалов 8, среднее время 0.02 мс
Ускорение: 8.0%


In [ ]:
# model.to(device)
# win_rate, avg_pure, avg_model = test_step()

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from pathlib import Path
import numpy as np
import time
from spatialmath import SE3
import roboticstoolbox as rtb

chunk_dir = Path('dataset/panda_dataset/tmp_chunks')
test_data_list = []
for start_idx in range(120000, 121000, 20): 
    fname = f'chunk_{start_idx}-{start_idx+19}.pt'
    fpath = chunk_dir / fname
    if fpath.exists():
        chunk = torch.load(fpath, weights_only=False)
        test_data_list.extend(chunk)
        print(f'{fname}: {len(chunk)} примеров')
    else:
        print(f'Файл {fpath} не найден!')

print(f'Всего тестовых примеров: {len(test_data_list)}')

# 2. DataLoader
test_loader = DataLoader(test_data_list, batch_size=64, shuffle=False)

# 3. Статистики для центрирования (должны быть уже посчитаны, иначе пересчитай)
# pos_mean, rpy_mean – тензоры на device

# 4. Функция тестирования
def test_step():
    robot = rtb.models.DH.Panda()
    model.eval()
    
    total = 0
    model_wins = 0
    pure_fails = 0
    model_fails = 0
    total_time_pure = 0.0
    total_time_model = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            
            # Сохраняем оригинальную цель
            orig_global_feat = batch_data.global_feat.clone()
            
            # Центрируем цель для модели
            gf = batch_data.global_feat
            gf[:, :3] -= pos_mean.to(device)
            gf[:, 3:6] -= rpy_mean.to(device)
            
            # Предсказание q_goal
            q_pred_batch = model(batch_data, q_goal=None)   # (B, 7)
            
            B = batch_data.num_graphs
            for i in range(B):
                total += 1
                # Целевая поза (ненормализованная)
                target_np = orig_global_feat[i].cpu().numpy()
                T_target = SE3(target_np[:3]) * SE3.RPY(target_np[3:6], order='zyx')
                
                # --- Чистый ikine_LM (500 итераций) из случайного старта в пределах сочленений ---
                q_start = np.random.uniform(robot.qlim[0], robot.qlim[1])
                t0 = time.perf_counter()
                try:
                    sol_pure = robot.ikine_LM(T_target, q0=q_start, ilimit=500, tol=1e-4, joint_limits=True)
                except Exception:
                    pure_fails += 1
                    continue
                t_pure = time.perf_counter() - t0
                
                # --- Модель + ikine_LM (200 итераций) ---
                q0_model = q_pred_batch[i].cpu().numpy()
                t0 = time.perf_counter()
                try:
                    sol_model = robot.ikine_LM(T_target, q0=q0_model, ilimit=200, tol=1e-4, joint_limits=True)
                except Exception:
                    model_fails += 1
                    continue
                t_model = time.perf_counter() - t0
                
                # Сравнение
                if not sol_pure.success:
                    pure_fails += 1
                    if sol_model.success:
                        model_wins += 1
                elif not sol_model.success:
                    model_fails += 1
                else:
                    total_time_pure += t_pure
                    total_time_model += t_model
                    if t_model < t_pure:
                        model_wins += 1
                        
    success_pairs = total - pure_fails - model_fails
    avg_pure = total_time_pure / success_pairs if success_pairs > 0 else 0.0
    avg_model = total_time_model / success_pairs if success_pairs > 0 else 0.0
    win_rate = model_wins / total if total > 0 else 0.0
    
    print(f"Всего примеров: {total}")
    print(f"Модель + IK быстрее в {win_rate*100:.1f}% случаев")
    print(f"Среднее время чистого IK (500 итер): {avg_pure*1000:.2f} мс")
    print(f"Среднее время IK с моделью (200 итер): {avg_model*1000:.2f} мс")
    print(f"Чистый IK не сошёлся: {pure_fails}")
    print(f"IK с моделью не сошёлся: {model_fails}")
    
    return win_rate, avg_pure, avg_model

# 5. Запуск теста
model.to(device)
test_step()


chunk_120000-120019.pt: 20 примеров
chunk_120020-120039.pt: 20 примеров
chunk_120040-120059.pt: 20 примеров
chunk_120060-120079.pt: 20 примеров
chunk_120080-120099.pt: 20 примеров
chunk_120100-120119.pt: 20 примеров
chunk_120120-120139.pt: 20 примеров
chunk_120140-120159.pt: 20 примеров
chunk_120160-120179.pt: 20 примеров
chunk_120180-120199.pt: 20 примеров
chunk_120200-120219.pt: 20 примеров
chunk_120220-120239.pt: 20 примеров
chunk_120240-120259.pt: 20 примеров
chunk_120260-120279.pt: 20 примеров
chunk_120280-120299.pt: 20 примеров
chunk_120300-120319.pt: 20 примеров
chunk_120320-120339.pt: 20 примеров
chunk_120340-120359.pt: 20 примеров
chunk_120360-120379.pt: 20 примеров
chunk_120380-120399.pt: 20 примеров
chunk_120400-120419.pt: 20 примеров
chunk_120420-120439.pt: 20 примеров
chunk_120440-120459.pt: 20 примеров
chunk_120460-120479.pt: 20 примеров
chunk_120480-120499.pt: 20 примеров
chunk_120500-120519.pt: 20 примеров
chunk_120520-120539.pt: 20 примеров
chunk_120540-120559.pt: 20 п

(0.611, 0.023583367100012312, 0.018933713600000374)

NameError: name 'test_loader' is not defined